# COMET InSAR Workshop: LiCSBAS Tutorial — **Module 0: dataset preparation (LiCSAR data)***A step-by-step tutorial to run LiCSBAS for the time series analysis. This first notebook covers **Module 0** (dataset preparation, steps `0-1` → `0-5`); the time series analysis (steps `11` → `16` + the interactive viewers) lives in [**`tutorial_1.ipynb`**](tutorial_1.ipynb).> **Other data routes for Module 0**> - [**NISAR data** (experimental)](tutorial_0_nisar.ipynb) — ALOS-2 L-band NISAR CSL2 GUNW via ASF (per frame)> - [**SBOVNL / SBOI data**](tutorial_0_sbovl.ipynb) — the `--sbovl` burst-overlap chain (steps `0-1` → `1-6`)

### please run the following cell first to load some necessary modules:  
(to run a cell, just click into it and press shift-Enter)  

#### Note: in case your notebook gets frozen/kernel disconnected, try clicking Kernel->Restart (rather than Refreshing page). If this runs ok, you may just rerun the following cell, and then skip back to the cell you were in before.

In [ ]:
import sys, os, subprocess, warnings
import importlib.util as _iu
from IPython.display import Image

def _run(cmd):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if r.returncode != 0:
        print((r.stdout or "")[-3000:])
        raise RuntimeError("command failed: " + cmd)

def _have(mod):
    try:
        return _iu.find_spec(mod) is not None
    except Exception:
        return False

# COMET local jupyterhub: add the legacy bash CLI wrappers (bin/) to PATH, if present
_hub = '/data/notebooks/repositories/LiCSBAS/bin'
if os.path.exists(_hub):
    os.environ['PATH'] = os.environ['PATH'] + ':' + _hub

# ---- LiCSBAS v3 is a pip package ('licsbas'). Make sure the (binary) GDAL stack
# ---- and the package itself are available before anything imports them.
# ---- This is idempotent: on a kernel that already has them, it is a no-op.
_installed = []

# 1) GDAL Python bindings (osgeo) + pyproj/rasterio are the hard binary deps of
#    licsbas. 'osgeo' is NOT on PyPI, so on an environment that has conda
#    (Colab, Binder) we pull it from conda-forge; otherwise it must already be
#    present in the system/conda env.
if not _have("osgeo"):
    if os.path.exists("/opt/conda/bin/conda") or _have("conda"):
        print("GDAL (osgeo) not found - installing from conda-forge (this can take a minute) ...")
        _run("conda install -y -q -c conda-forge gdal pyproj=3.6")
        _installed.append("gdal (conda-forge)")
    else:
        raise SystemExit(
            "GDAL Python bindings ('osgeo') are required by licsbas but not found.\n"
            "Install them, e.g., with:\n"
            "    conda install -c conda-forge gdal pyproj=3.6\n"
            "(on Google Colab this cell should auto-install them via conda-forge).")

# 2) the licsbas package itself
if not _have("licsbas"):
    print("licsbas not found - installing 'licsbas[viewer]' via pip ...")
    _run('pip install -q "licsbas[viewer]"')
    _installed.append("licsbas[viewer]")

warnings.filterwarnings('ignore')

if _installed:
    print("\n[setup] installed: " + ", ".join(_installed))
    print("[setup] New native libraries need a KERNEL RESTART to be picked up.")
    print("[setup]   -> Runtime > Restart session (or Restart kernel), then re-run THIS cell.\n")
    raise SystemExit("Please restart the kernel and re-run this cell to finish setup.")

# some extra functions
def display(png):
    if not os.path.exists(png):
        print('This png file does not exist, recheck please')
    else:
        return Image(png)

def printbold(instr='done'):
    print('\033[1m' + instr + '\033[0m')

# selected frame
frame = '022D_04826_121209'  # for tutorial

# locate the pre-extracted tutorial dataset (binder postBuild / install script)
framedir = None
for cand in (os.getcwd(),
             os.path.join(os.getcwd(), frame),
             os.path.join(os.environ.get('HOME', '.'), frame)):
    if os.path.exists(os.path.join(cand, 'GEOCml2GACOS')):
        framedir = cand
        break

if framedir is None:
    print('WARNING: tutorial dataset not found (GEOCml2GACOS).')
    print('Download and extract it first - see tutorial/install_complete.sh or the binder postBuild script.')
else:
    os.chdir(framedir)

print('prerequisites loaded, thank you. Running in directory:')
print(os.getcwd())
printbold()


# LiCSBAS Tutorial

LiCSBAS is an open-source package in Python and bash to carry out InSAR time series analysis using LiCSAR interfeometric products (primarily unwrapped interferograms and coherence) which are freely available on the [COMET-LiCS web portal](https://comet.nerc.ac.uk/COMET-LiCS-portal/).



Users can easily and automatically derive the time series and velocity of the displacement if sufficient LiCSAR products are available in the area of interest. LiCSBAS also contains visualization tools to interactively display the time series of displacement to help investigation and interpretation of the results. Current release offers various improvements including option to perform customized unwrapping procedure using *LiCSBAS02to05_unwrap.py*.  

## Documentation

The [COMET LiCSBAS repository](https://github.com/comet-licsar/licsbas) is an actively developed fork of the [original LiCSBAS](https://github.com/yumorishita/LiCSBAS) tool developed by Dr. Yu Morishita during his stay at the University of Leeds. For more detailed information, see the [**wiki**](https://github.com/yumorishita/LiCSBAS/wiki) pages and [quick start](https://github.com/yumorishita/LiCSBAS/wiki/2_0_workflow#quick-start).  

This tutorial is run through Jupyter Notebook - see short [tutorial](https://www.dataquest.io/blog/jupyter-notebook-tutorial) if you are not familiar with this environment.

## Standard LiCSBAS Workflow

<img src="https://raw.githubusercontent.com/wiki/yumorishita/LiCSBAS/images/workflow.png" width=650>

**Note:** Current workflow is extended by several optional processes, such as advanced selection of reference point (LiCSBAS120 by Dr. Qi Ou), pixel-based nullification of unwrapping errors (part of LiCSBAS12, by Dr. Lin Shen), advanced unwrapping procedure (LiCSBAS02to05, by Dr. Milan Lazecky), estimation of non-linear model and offsets (LiCSBAS_cum2vel, by Dr. Yu Morishita and M. Lazecky), ability to load ALOS interferograms (LiCSBAS01_get_geotiff_ALOS included from Dr. Yu Morishita's updated solutions), among various other changes. As we keep using standard LiCSBAS functionality within this tutorial, we did not update flowchart and other figures and leave them for investigation of individual parameters of the LiCSBAS scripts.

# Module 0: Preparing the dataset

This tutorial will run on a preprocessed dataset (i.e. modules 0 are complete here) that is clipped in both time and space, and cover covers volcanic area of Campi Flegrei in Napoli region (Italy). The figure below shows GPS network measurements over the Napoli region, published in https://www.mdpi.com/2072-4292/13/14/2725. The Campi Flegrei caldera (in the middle) has been inflating since 2010s, a continuous uplift can be observed from the GPS station time series (middle plot).  

<img src="tutorial/campi_flegrei_gps.png" width="1000">

You may use the following module 0 steps to process another frame of your interest. **For the purposes of this tutorial, they are marked as skipped**.  


Note: you may identify LiCSAR frames covering your own area of interest (AOI) in your free time later. For this, you may just check the [COMET LiCSAR portal](https://comet.nerc.ac.uk/COMET-LiCS-portal/).  


## The processing chain of this module (steps `0-1` → `0-5`, with the **02to05** alternative)<img src="https://raw.githubusercontent.com/comet-licsar/LiCSBAS/towards-v3/tutorial/0_licsar.png" width="100%">*(image not visible? the PNGs also live in `tutorial/0_licsar.png`; `python3 tutorial/make_flowchart.py` regenerates them)*

## (skipping) Step 0-1: Download GeoTIFF from COMET-LiCS web portal


### licsbas-01-get-geotiff
This script downloads GeoTIFF files of unw (unwrapped interferogram) and cc (coherence) in the specified frame ID from COMET-LiCS web portal.  
The -f option is not necessary when the frame ID can be automatically identified from the name of the working directory.  
GACOS data can also be downloaded if available, as well as phase data (and intensity) required for the lately added unwrapping procedure.  
Existing GeoTIFF files are not re-downloaded to save time, i.e., only the newly available data will be downloaded.  

**Note:** LiCSBAS is capable of running on any network of standard interferograms if these exist in GEOC/YYYYMMDD_YYYYMMDD together with coherence (either in 0-1 or 0-255 range)

**cell \[ 01 \]**

In [ ]:
! licsbas-01-get-geotiff -h
#e.g. licsbas-01-get-geotiff -f 167D_04884_131212 --get_gacos --n_para 1
#printbold()

## (skipping) Step 0-2: Convert and downsample

### licsbas-02-ml-prep
This script converts GeoTIFF files of unw and cc to float32 and uint8 format, respectively, for further time series analysis, and also downsamples (multilooks) data if specified.  
Existing files are not re-created to save time, i.e., only the newly available data will be processed.  

**Note:** Use of LiCSBAS02to05_unwrap.py replaces this and following 0-x optional steps, however they still can be applied to its output.

**cell \[ 02 \]**

In [ ]:
! licsbas-02-ml-prep -h
# e.g. licsbas-02-ml-prep -i GEOC -o GEOCml10 -n 10 --npara 1
printbold()

## Alternative route: **Step `02to05` — re-unwrapping (substitutes steps `0-2` → `0-5`)**So far we have consumed *already-unwrapped* data (the normal case, where the interferograms were unwrapped by the data provider).`LiCSBAS02to05_unwrap.py` is an **alternative route** for cases where the input interferograms are only **wrapped** (`.geo.diff..._pha.tif`, downloaded with `--get_pha`): it performs **the unwrapping itself** and, in a single invocation, carries out *all* of steps `0-2` (multilooking), `0-3` (GACOS, if `--gacos`), `0-4` (masking) and `0-5` (clipping) — i.e. it **substitutes steps `02`–`05`** (see the amber box in the flowchart above).It unwraps (Goldstein filter, stability/constancy threshold `--thres`, optional cascade re-unwrapping `--cascade`/`--cascade_full`), an optional height-correlation correction (`--hgtcorr`), and produces the same `GEOCml{n}[GACOS][mask][clip]` layout as the standard route (incl. `coherence`, `unw`, `baselines`, `par` files).> Requires the `unwrap` extra: `pip install licsbas[unwrap]` (uses SNAPHU for some filters).

In [ ]:
![-h help] licsbas-02to05-unwrap -h

In [ ]:
# EXAMPLE (not run in this tutorial)# licsbas-02to05-unwrap -i GEOCdir -M 5 --gacos --cascade --thres 0.3# licsbas-02to05-unwrap -i GEOCdir -M 10 --filter gauss -g 14.00/14.25/40.75/40.95   # + clip

*In this tutorial we continue with the standard (already-unwrapped) route.*

# (skipping) Step 0-3: GACOS correction (optional)

### licsbas-03op-gacos
This script applies a tropospheric correction to unw data using GACOS data.  
GACOS data may be automatically downloaded from COMET-LiCS web at step01 (if available), or could be externally obtained by requesting on a [GACOS web](http://www.gacos.net/).  
(if you request the GACOS data through the GACOS web, the dates and time of interest can be found in baselines and slc.mli.par, respectively. Note that original GACOS data comes as zenith total delay, ZTD. This is automatically converted to the slant direction of the satellite, i.e. into slant total delay, SLTD).  

 
The impact of the correction can be visually checked by showing GACOS_info.png and \*/\*.gacos.png:  
- unw_org - original unwrapped data (STD = overall standard deviation of all unw values)
- unw_cor - unwrapped data after GACOS correction
- dsltd - differential (respective to the interferometric pair) correction image from GACOS ZTD data converted to SLTD (see above)

<img src="https://raw.githubusercontent.com/wiki/yumorishita/LiCSBAS/images/20170407_20170501.gacos.png" width=600>  

<img src="https://raw.githubusercontent.com/wiki/yumorishita/LiCSBAS/images/GACOS_info.png" width=600>

**cell \[ 03 \]**

In [ ]:
! licsbas-03op-gacos -h
# e.g. licsbas-03op-gacos -i GEOCml10 -o GEOCml10GACOS --fillhole --n_para 1
printbold()

## (skipping) Step 0-4: Mask UNW (optional)

### licsbas-04op-mask-unw
This script masks specified areas or low coherence areas in the unw data.  
The masking is effective when the unw data include areas which have many unwrapping errors and are not of interest, and can improve the result of Step 1-2 (loop closure).   

<img src="https://raw.githubusercontent.com/wiki/yumorishita/LiCSBAS/images/20150524_20150617.unw_mask.png" width=600>

**Note:** In practice, an option to mask data based on their individual coherence (parameter *-s*) is valuable for reliable inversion in the later steps.

**cell \[ 04 \]**

In [ ]:
! licsbas-04op-mask-unw -h
printbold()

## (skipping) Step 0-5: Clip UNW (optional)

### licsbas-05op-clip-unw
This script clips a specified rectangular area of interest from unw and cc data.  
The clipping can make the data size smaller and processing faster, and improve the quality measures calculated for the whole scene, such as avg loop phase closure result of Step 1-2.  

<img src="https://raw.githubusercontent.com/wiki/yumorishita/LiCSBAS/images/20150524_20150617.unw_clip.png" width=400>

**cell \[ 05 \]**

In [ ]:
! licsbas-05op-clip-unw -h
# e.g. licsbas-05op-clip-unw -i GEOCml10GACOS -o GEOCml10GACOSclip -g 14.03/14.22/40.78/40.90 --n_para 1
printbold()

# Running steps 11-16 using existing preprocessed data

LiCSBAS processing directory has following folders:
- GEOC - location for the original data, (auto-)downloaded from COMET LiCSAR
- GEOCmlX - result of Step 0-2 - X-times downsampled converted data
- GEOCmlXGACOS - optional - result of GACOS correction (steps 0-4 and 0-5 also generate optional 'final pre-processed dataset'
- GEOCmlX\*clip - optional - spatially clipped dataset
- TS_GEOC* - folder containing outputs of time series processing (see further)

**Note:** We have preprocessed data for the given frame for you, for the purposes of this tutorial.

**cell \[ 06 \]**

In [ ]:
# check the contents of the folder
! pwd
! ls


# check the number of interferograms
! echo The folder contains `ls GEOCml2GACOSclip | wc -l` interferograms
! echo "last 30 folders+files:"
! ls GEOCml2GACOSclip | tail -n 30

In [ ]:
**cell \[ 07 \]**

# check the interferograms network:
! ls GEOCml2GACOSclip | grep '_20' > ifg_list.txt
print('generating network plot')
! licsbas-plot-network -i ifg_list.txt -b GEOCml2GACOSclip/baselines -o network.png >/dev/null 2>/dev/null
print('first 10 interferogram pairs:')
! head -n 10 ifg_list.txt
display('network.png')

In [ ]:
**cell \[ 08 \]**

# look into some random interferogram:
pair = '20161014_20161026'
png = os.path.join('GEOCml2GACOSclip',pair,pair+'.unw.png')
display(png)

In [ ]:
**cell \[ 09 \]**

In [ ]:
# see effect of GACOS of this selected interferogram:
# note GACOS corrections are created before clipping, thus they are in the second extracted folder
pair='20161008_20161026'
#pair='20161002_20161020'  # pair causing higher std after GACOS
print('unw_org - original unwrapped data')
print('unw_cor - corrected unw data')
print('dsltd - differential slant direction total delay estimate from GACOS data')
print('colour scale: 6pi/colour cycle')
png = os.path.join('GEOCml2GACOS',pair,pair+'.gacos.png')
display(png)

In [ ]:
# look into some interferogram - e.g. see effect of GACOS:
! head -n 10 GEOCml2GACOSclip/GACOS_info.txt
png = 'GEOCml2GACOSclip/GACOS_info.png'
display(png)

## Next step: Module 1 — Time series (steps `11` → `16` + viewers)**[Continue in `tutorial_1.ipynb` (opens in this Binder session) &raquo;](<tutorial_1.ipynb>)**- [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/comet-licsar/LiCSBAS/HEAD?labpath=tutorial/tutorial_1.ipynb) - *new Binder session*- [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/comet-licsar/LiCSBAS/blob/towards-v3/tutorial/tutorial_1.ipynb) - *Colab**Alternatively, jump to the [NISAR data route](tutorial_0_nisar.ipynb) or the [SBOVNL/SBOI route](tutorial_0_sbovl.ipynb).*